# Домашнее задание №15. Keras

## Задание 1
1.Самостоятельно выбранными средствами (opencv, pillow (PIL), …) сгенерировать по 820 картинок размером 100х100 пикселей (px) для каждой из цифр: 0, 1, 3, 8 следующим образом (800 – тренировочная выборка, 20 – тестовая выборка № 1):
 фон картинки белый,\
    1. цифра: ширина – 20 px, высота – 50 px, цвет линии – черный, цифра целиком помещается в картинку, цифра находится в случайном месте на картинке,\
    2. на изображении цифра расположена так, что ее вертикальная ось параллельна оси ординат (вертикальное положение) или оси абсцисс (горизонтальное положение),\
    3. тренировочная выборка содержит 400 изображений каждой цифры в горизонтальном положении и 400 изображений каждой цифры в вертикальном положении,\
    4. тестовая выборка содержит 10 изображений каждой цифры в горизонтальном положении и 10 изображений каждой цифры в вертикальном положении.

2.Создать новые тестовые картинки, полученные путем добавления черных пикселей (шум) в случайно выбранные места сгенерированных тестовых картинок:\
    1. 20 пикселей (тестовая выборка № 2),\
    2. 50 пикселей (тестовая выборка № 3),\
    3. 100 пикселей (тестовая выборка № 4),\
    4. 200 пикселей (тестовая выборка № 5).

In [ ]:
import os
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import random

IMAGE_SIZE = 100
DIGIT_WIDTH = 20
DIGIT_HEIGHT = 50
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)

def create_dirs():
    digits = ['0', '1', '3', '8']
    sets = ['train', 'test1', 'test2', 'test3', 'test4', 'test5','test6']
    
    for digit in digits:
        for set_name in sets:
            os.makedirs(f'C:/Users/reino/python/hw/data/{set_name}/{digit}', exist_ok=True)

def generate_digit_image(digit, orientation):
    img = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE), WHITE)
    draw = ImageDraw.Draw(img)
    
    try:
        font = ImageFont.truetype("arial.ttf", size=max(DIGIT_WIDTH, DIGIT_HEIGHT))
    except:
        raise ValueError("Font not found")

    bbox = draw.textbbox((0, 0), digit, font=font)
    digit_w = bbox[2] - bbox[0]
    digit_h = bbox[3] - bbox[1]
    
    scale_w = DIGIT_WIDTH / digit_w
    scale_h = DIGIT_HEIGHT / digit_h
    scale = min(scale_w, scale_h)
    
    new_w = int(digit_w * scale)
    new_h = int(digit_h * scale)

    if orientation == 'horizontal':
        max_x = IMAGE_SIZE - new_w
        max_y = IMAGE_SIZE - new_h
        x = random.randint(0, max_x)
        y = random.randint(0, max_y)
    else:
        img = img.rotate(90, expand=True)
        draw = ImageDraw.Draw(img)
        max_x = IMAGE_SIZE - new_h
        max_y = IMAGE_SIZE - new_w
        x = random.randint(0, max_x)
        y = random.randint(0, max_y)
    
    draw.text((x, y), digit, font=font, fill=BLACK)
    
    if orientation == 'vertical':
        img = img.rotate(-90, expand=True)
    
    return img

def add_noise(img, noise_pixels):
    img_array = np.array(img)
    height, width = img_array.shape[:2]
    
    for _ in range(noise_pixels):
        x = random.randint(0, width-1)
        y = random.randint(0, height-1)
        img_array[y, x] = BLACK
    
    return Image.fromarray(img_array)

def generate_noisy_test_sets():
    digits = ['0', '1', '3', '8']
    orientations = ['horizontal', 'vertical']
    noise_levels = {
        'test2': 20,
        'test3': 50,
        'test4': 100,
        'test5': 200,
        'test6': 1000
    }
    
    for digit in digits:
        for orientation in orientations:
            for i in range(10):
                original_path = f'C:/Users/reino/python/hw/data/test1/{digit}/{digit}_{orientation}_{i:03d}.png'
                img = Image.open(original_path)
                for test_name, pixels in noise_levels.items():
                    noisy_img = add_noise(img.copy(), pixels)
                    noisy_img.save(f'C:/Users/reino/python/hw/data/{test_name}/{digit}/{digit}_{orientation}_{i:03d}.png')

def generate_dataset():
    digits = ['0', '1', '3', '8']
    orientations = ['horizontal', 'vertical']
    
    create_dirs()
    for digit in digits:
        for orientation in orientations:
            for i in range(400):
                img = generate_digit_image(digit, orientation)
                img.save(f'C:/Users/reino/python/hw/data/train/{digit}/{digit}_{orientation}_{i:03d}.png')

    for digit in digits:
        for orientation in orientations:
            for i in range(10):
                img = generate_digit_image(digit, orientation)
                img.save(f'C:/Users/reino/python/hw/data/test1/{digit}/{digit}_{orientation}_{i:03d}.png')

    generate_noisy_test_sets()

if __name__ == "__main__":
    generate_dataset()
    print("Все изображения успешно сгенерированы!")

Все изображения успешно сгенерированы!


# Задание №2

Не используя предобученные модели (сети), модифицировать скрипт задачи «Dogs vs Cats» (с семинара) или написать свою нейронную сеть на keras такую, что:

1) На вход подается тренировочное множество: по 800 картинок каждой цифры.
2) Из тренировочного множества выделяется часть картинок (10-20%), на валидационное множество, в котором должны присутствовать цифры в вертикальном и горизонтальном положении.
3) Протестировать адекватность модели на всех тестовых выборках № 1, № 2, № 3, № 4, № 5, фиксируя при этом точность (accuracy) классификации.
4) Повторить пункты 1)–3), изменив объем тренировочной выборки до 600, 400, 200, 100 картинок каждой цифры.

<i> Могут пригодиться <tt>Dense</tt>, <tt>Conv2D</tt>, <tt>MaxPooling2D</tt>, <tt>Flatten</tt>.</i>

Результаты оформить в виде таблицы со столбцами: размер тренировочной выборки, количество шумовых пикселей, точность (accuracy) классификации.

In [ ]:
import os
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
import pandas as pd
from PIL import Image
import shutil

IMG_SIZE = 100
BATCH_SIZE = 32
EPOCHS = 30
CLASSES = ['0', '1', '3', '8']
TEST_SETS = {
    'test1': 0,
    'test2': 20,
    'test3': 50,
    'test4': 100,
    'test5': 200,
    'test6': 1000
}

def create_model():
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        MaxPooling2D(2, 2),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        Conv2D(128, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        Flatten(),
        Dense(512, activation='relu'),
        Dense(len(CLASSES), activation='softmax')
    ])
    
    model.compile(optimizer='adam',
                 loss='categorical_crossentropy',
                 metrics=['accuracy'])
    return model

def prepare_subset(source_dir, target_dir, subset_size):
    os.makedirs(target_dir, exist_ok=True)
    for digit in CLASSES:
        os.makedirs(os.path.join(target_dir, digit), exist_ok=True)
        files = os.listdir(os.path.join(source_dir, digit))
        selected_files = np.random.choice(files, subset_size, replace=False)
        for file in selected_files:
            shutil.copy(os.path.join(source_dir, digit, file),
                       os.path.join(target_dir, digit, file))

def load_data(train_dir, test_dirs, validation_split=0.2):
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        validation_split=validation_split,
        rotation_range=10,
        width_shift_range=0.1,
        height_shift_range=0.1
    )
    
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='training'
    )
    
    val_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='validation'
    )
    
    test_generators = {}
    for test_name, test_dir in test_dirs.items():
        test_datagen = ImageDataGenerator(rescale=1./255)
        test_generators[test_name] = test_datagen.flow_from_directory(
            test_dir,
            target_size=(IMG_SIZE, IMG_SIZE),
            batch_size=BATCH_SIZE,
            class_mode='categorical',
            shuffle=False
        )
    
    return train_generator, val_generator, test_generators

def train_and_evaluate(train_sizes):
    results = []
    base_train_dir = 'C:/Users/reino/python/hw/data/train'
    test_dirs = {name: f'C:/Users/reino/python/hw/data/{name}' for name in TEST_SETS.keys()}
    
    for size in train_sizes:
        train_subset_dir = f'C:/Users/reino/python/hw/data/train_{size}'
        prepare_subset(base_train_dir, train_subset_dir, size//len(CLASSES))
        train_gen, val_gen, test_gens = load_data(train_subset_dir, test_dirs)
        model = create_model()
        early_stopping = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)
        
        history = model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=EPOCHS,
            callbacks=[early_stopping],
            verbose=1
        )
        
        for test_name, test_gen in test_gens.items():
            _, accuracy = model.evaluate(test_gen, verbose=0)
            results.append({
                'Размер выборки': size,
                'Шум (пикс)': TEST_SETS[test_name],
                'Точность': f"{accuracy:.4f}"
            })
    
    return pd.DataFrame(results)

def main():
    train_sizes = [800, 600, 400, 200, 100]
    results_df = train_and_evaluate(train_sizes)
    print(results_df.to_string(index=False))
    results_df.to_csv('classification_results.csv', index=False)

if __name__ == "__main__":
    main()

Found 640 images belonging to 4 classes.
Found 160 images belonging to 4 classes.
Found 80 images belonging to 4 classes.
Found 80 images belonging to 4 classes.
Found 80 images belonging to 4 classes.
Found 80 images belonging to 4 classes.
Found 80 images belonging to 4 classes.
Found 80 images belonging to 4 classes.


C:\Users\reino\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\reino\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 156ms/step - accuracy: 0.2328 - loss: 1.7313 - val_accuracy: 0.2500 - val_loss: 1.3923
Epoch 2/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 89ms/step - accuracy: 0.2500 - loss: 1.3841 - val_accuracy: 0.2750 - val_loss: 1.3778
Epoch 3/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 88ms/step - accuracy: 0.2742 - loss: 1.3409 - val_accuracy: 0.2562 - val_loss: 1.3689
Epoch 4/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 88ms/step - accuracy: 0.3928 - loss: 1.2665 - val_accuracy: 0.3250 - val_loss: 1.3194
Epoch 5/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 88ms/step - accuracy: 0.4828 - loss: 1.2014 - val_accuracy: 0.3875 - val_loss: 1.2488
Epoch 6/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 91ms/step - accuracy: 0.5832 - loss: 1.0835 - val_accuracy: 0.5375 - val_loss: 1.0929
Epoch 7/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 89ms/step - accuracy: 0.6402 - loss: 0.8903 - val_accuracy: 0.5250 - val_loss: 0.9890
Epoch 8/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 89ms/step - accuracy: 0.6892 - loss: 0.7692 - val_accuracy: 0.6562 - 